# Step 2: build the LoRA dataset for a Jev-style tool router

Turns **xLAM-60k** (Salesforce, CC-BY-4.0: real tool schemas + the correct call with its arguments) into four kinds of routing decision, following the same recipe NVIDIA used to build **When2Call** from BFCL:

| Label | How it's made | Share |
|---|---|---|
| `request_for_info` (**ask**) | delete a required detail the user gave (e.g. "factorial **of 5**" → "factorial"); single-call requests only | 30% |
| `cannot_answer` (**decline**) | remove the correct tool; keep the other tools plus look-alike "near-miss" tools; sometimes no tools at all | 25% |
| `tool_call` (**call**) | the original request with its tools | 35% |
| `direct` (**answer directly**) | a self-contained instruction from Alpaca that needs no tool, with unrelated tools listed | 10% |

On top of these routing rows, **general decisions** from [Open-Jev](https://huggingface.co/datasets/ZefanCai/Open-Jev) (CC0: yes/no, multiple-choice and rating questions over tickets, policies and other synthetic states) are mixed in as about a quarter of the data, so the model keeps (and improves) the general decision skill Kev's transfer-v4 test measures instead of forgetting it while learning routing.

Every example is written in exactly the prompt formats the evaluation notebook uses:
- **action** style: Kev/When2Call's four-way question "What should the assistant do next?"
- **route** style: our BFCL question, lettered tools + `NONE` (ask / decline / direct → `NONE`, following BFCL, which treats missing details as "no call"). Call examples go to this style more often so it stays about half "call a tool", half `NONE`, like the BFCL test.

Nothing from BFCL, When2Call or Kev's suites is used for training, and any training row whose text matches an evaluation item is removed.

### Run it on Kaggle (CPU is enough, ~5 min)
1. Open https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k while logged in and click **Agree** to its terms.
2. Create a read token at https://huggingface.co/settings/tokens.
3. In Kaggle: **Add-ons → Secrets → Add secret**, name `HF_TOKEN`, paste the token, and tick it for this notebook. **Settings → Internet → On.**
4. **Run All.** Output: `/kaggle/working/lora_data/train.jsonl`, `val.jsonl`, `stats.json`.

In [ ]:
N_EXAMPLES = 12000                 # base examples; each becomes one training row
MIX = {"request_for_info": 0.30, "cannot_answer": 0.25, "tool_call": 0.35, "direct": 0.10}
ROUTE_SHARE = {"tool_call": 0.5, "request_for_info": 0.27, "cannot_answer": 0.27, "direct": 0.27}   # rest -> action style
SHUFFLE_ACTION_OPTIONS = 0.5       # share of action rows with the four options in a random order (eval uses the fixed order)
EMPTY_TOOLS_SHARE = 0.15           # share of decline rows with no tools at all (When2Call has these)
MAX_TOOLS = 6
DESC_CHARS = 300                   # same trimming as the evaluation notebook
GENERAL_SHARE = 0.25              # share of all rows that are Open-Jev general decisions (0 = routing only)
OPENJEV_CONFIG = "release-v2-redistributable"
EXCLUDE_GENERAL = r"vizdoom|snake|tile_platformer|tic_tac_toe|trex|painting|chess|tetris|minesweeper|drone|browser|game"   # game/control sources
VAL_FRACTION = 0.05
SEED = 42
OUT_DIR = "/kaggle/working/lora_data"

KEV_COMMIT = "1f04cfa28e6a708485b28a9585010b9e436b776e"
BFCL_COMMIT = "6ea57973c7a6097fd7c5915698c54c17c5b1b6c8"
KEV_RAW = f"https://raw.githubusercontent.com/jaredpalmer/kev/{KEV_COMMIT}/"
BFCL_RAW = (f"https://raw.githubusercontent.com/ShishirPatil/gorilla/{BFCL_COMMIT}/"
            "berkeley-function-call-leaderboard/bfcl_eval/data/")

In [ ]:
import os, re, json, random, collections, urllib.request

os.makedirs(OUT_DIR, exist_ok=True)
rng = random.Random(SEED)


def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")


def load_sources():
    from datasets import load_dataset
    xlam = load_dataset("Salesforce/xlam-function-calling-60k", split="train", token=hf_token())
    alpaca = load_dataset("tatsu-lab/alpaca", split="train")
    openjev = []
    if GENERAL_SHARE > 0:
        try:
            openjev = list(load_dataset("ZefanCai/Open-Jev", OPENJEV_CONFIG, split="train"))
        except Exception as e:
            print(f"Open-Jev not loaded ({type(e).__name__}: {e}); continuing with routing rows only")
    return list(xlam), list(alpaca), openjev


def fetch_text(url, cache):
    path = os.path.join(OUT_DIR, "cache", cache)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return open(path, encoding="utf-8").read()


# Kev's renderer: the action-style prompt must match the evaluation exactly.
kev_api = fetch_text(KEV_RAW + "kev/api.py", "kev_api.py")
os.makedirs(os.path.join(OUT_DIR, "cache", "kev"), exist_ok=True)
open(os.path.join(OUT_DIR, "cache", "kev", "__init__.py"), "a").close()
open(os.path.join(OUT_DIR, "cache", "kev", "api.py"), "w").write(kev_api)
import sys
sys.path.insert(0, os.path.join(OUT_DIR, "cache"))
from kev.api import render, option_text

xlam_rows, alpaca_rows, openjev_rows = load_sources()
print(len(xlam_rows), "xLAM rows,", len(alpaca_rows), "Alpaca rows,", len(openjev_rows), "Open-Jev rows")

## Convert xLAM tools to the BFCL schema the evaluation uses

In [ ]:
TYPE_MAP = {"str": "string", "string": "string", "int": "integer", "integer": "integer", "float": "float",
            "number": "float", "bool": "boolean", "boolean": "boolean", "dict": "dict", "object": "dict",
            "list": "array", "array": "array", "tuple": "tuple", "any": "any"}


def norm_type(t):
    t = str(t or "any").split(",")[0].strip()          # "str, optional" -> "str"
    base = re.split(r"[\[\s]", t)[0].lower()            # "List[int]" -> "list"
    return TYPE_MAP.get(base, "any")


def to_bfcl(tool):
    # xLAM: {"name", "description", "parameters": {param: {"description", "type", "default"?}}}
    params = tool.get("parameters") or {}
    if "properties" in params:                         # already JSON-schema shaped
        props = params.get("properties") or {}
        required = list(params.get("required") or [])
    else:
        props, required = {}, []
        for pname, spec in params.items():
            spec = spec if isinstance(spec, dict) else {}
            props[pname] = {"type": norm_type(spec.get("type")), "description": str(spec.get("description", ""))}
            if "default" in spec:
                props[pname]["default"] = spec["default"]
            optional = "default" in spec or "optional" in str(spec.get("type", "")).lower()
            if not optional:
                required.append(pname)
    return {"name": tool["name"], "description": str(tool.get("description", "")),
            "parameters": {"type": "dict", "required": required, "properties": props}}


def required_params(fn):
    return list(fn["parameters"].get("required") or [])


base = []
for i, row in enumerate(xlam_rows):
    try:
        tools = [to_bfcl(t) for t in json.loads(row["tools"])]
        answers = json.loads(row["answers"])
    except (json.JSONDecodeError, KeyError, TypeError):
        continue
    if not tools or not answers:
        continue
    names = {a["name"] for a in answers}
    if len(names) != 1 or not any(t["name"] == next(iter(names)) for t in tools):
        continue                                       # keep requests that need exactly one kind of tool
    if len({t["name"] for t in tools}) != len(tools):
        continue
    base.append({"id": f"xlam-{i}", "query": row["query"].strip(), "tools": tools, "gold": next(iter(names)),
                 "args": answers[0].get("arguments") or {}, "n_calls": len(answers)})
print(len(base), "usable xLAM requests (one kind of tool, gold tool listed)")

# index of all tools by name token, for look-alike "near-miss" tools
all_tools = {}
for b in base:
    for t in b["tools"]:
        all_tools.setdefault(t["name"], t)
token_index = collections.defaultdict(list)
name_tokens = lambda n: {w for w in re.split(r"[^a-z0-9]+", re.sub(r"([a-z])([A-Z])", r"\1 \2", n).lower()) if len(w) >= 4}
for n in all_tools:
    for w in name_tokens(n):
        token_index[w].append(n)

## Build the four kinds of example

In [ ]:
CONNECTORS = r"(?:of|for|in|at|to|from|with|on|by|named|called|titled|is|as|about|using|=|:)"
REALTIME = re.compile(r"\b(current|currently|today|tomorrow|latest|now|weather|price|stock|news|live|this week|tonight)\b", re.I)


def desc_words(t):
    return set(re.findall(r"[a-z]{4,}", t["description"].lower()))


def jaccard(a, b):
    return len(a & b) / max(1, len(a | b))


def make_call(b):
    return {"variant": "tool_call", "query": b["query"], "tools": b["tools"], "gold_tool": b["gold"]}


UNITS = (r"(?:\s*(?:cm|mm|m|km|kg|g|mg|lbs?|pounds|miles|meters|metres|feet|ft|inches|seconds|secs|minutes|mins|"
         r"hours|hrs|days|weeks|months|years|percent|%|degrees|units|dollars|usd|eur|gb|mb|items|people))?\b")


def make_ask(b):
    if b["n_calls"] != 1:                              # with several calls, removing one value leaves others callable
        return None
    gold = next(t for t in b["tools"] if t["name"] == b["gold"])
    candidates = []
    for p in required_params(gold):
        v = b["args"].get(p)
        if v is None or isinstance(v, (dict, list, bool)):
            continue
        v = str(v).strip()
        if len(v) < 2 and not v.isdigit():
            continue
        if re.search(re.escape(v), b["query"], re.I):
            candidates.append(v)
    rng.shuffle(candidates)
    for v in candidates:
        unit = UNITS if re.fullmatch(r"-?\d+(?:\.\d+)?", v) else ""
        pat = re.compile(rf"\s*(?:\b{CONNECTORS}\s+)?(?:the\s+)?['\"]?{re.escape(v)}['\"]?{unit}", re.I)
        q = pat.sub("", b["query"], count=1)
        q = re.sub(r"\s+([,.?!])", r"\1", re.sub(r"\s{2,}", " ", q)).strip(" ,")
        q = re.sub(r"\(\s*\)|,\s*,", "", q).strip()
        if re.search(re.escape(v), q, re.I) or len(q.split()) < 4:
            continue
        return {"variant": "request_for_info", "query": q, "tools": b["tools"], "gold_tool": None, "removed": v}
    return None


def near_miss(gold):
    gw, gd = name_tokens(gold["name"]), desc_words(gold)
    pool = {n for w in gw for n in token_index[w]} - {gold["name"]}
    picks = [all_tools[n] for n in pool if jaccard(desc_words(all_tools[n]), gd) < 0.35]
    rng.shuffle(picks)
    return picks[:2]


def make_decline(b):
    gold = next(t for t in b["tools"] if t["name"] == b["gold"])
    if rng.random() < EMPTY_TOOLS_SHARE:
        return {"variant": "cannot_answer", "query": b["query"], "tools": [], "gold_tool": None}
    gd = desc_words(gold)
    rest = [t for t in b["tools"] if t["name"] != gold["name"] and jaccard(desc_words(t), gd) < 0.35]
    tools = rest + [t for t in near_miss(gold) if t["name"] not in {x["name"] for x in rest}]
    if not tools:
        return None
    return {"variant": "cannot_answer", "query": b["query"], "tools": tools[:MAX_TOOLS], "gold_tool": None}


def make_direct(a):
    if a.get("input") or REALTIME.search(a["instruction"]) or len(a["instruction"].split()) < 4:
        return None
    tools = rng.sample(list(all_tools.values()), rng.randint(1, 4))
    return {"variant": "direct", "query": a["instruction"].strip(), "tools": tools, "gold_tool": None}


targets = {k: round(v * N_EXAMPLES) for k, v in MIX.items()}
order = base[:]
rng.shuffle(order)
built = collections.defaultdict(list)
makers = {"request_for_info": make_ask, "cannot_answer": make_decline, "tool_call": make_call}
for b in order:
    open_kinds = [k for k in makers if len(built[k]) < targets[k]]
    rng.shuffle(open_kinds)                            # no kind gets first pick of the source requests
    for kind in open_kinds:
        ex = makers[kind](b)
        if ex:
            ex["source_id"] = b["id"]
            built[kind].append(ex)
            break                                      # one variant per source request, so splits stay clean
    if all(len(built[k]) >= targets[k] for k in ("request_for_info", "cannot_answer", "tool_call")):
        break
alp = alpaca_rows[:]
rng.shuffle(alp)
for i, a in enumerate(alp):
    if len(built["direct"]) >= targets["direct"]:
        break
    ex = make_direct(a)
    if ex:
        ex["source_id"] = f"alpaca-{i}"
        built["direct"].append(ex)
print({k: f"{len(v)}/{targets[k]}" for k, v in built.items()})

## Render in the evaluation's exact prompt formats

In [ ]:
LETTERS = [chr(ord("A") + i) for i in range(26)]
# --- route style: copied from the evaluation notebook ---
SYSTEM = ("You are a tool-routing decision model. Read the conversation and the lettered options, "
          "then choose the ONE option that should happen next: the single tool to call, or NONE if "
          "no listed tool can fulfil the request. Reply with the option letter only.")


def render_tool(fn):
    params = fn.get("parameters") or {}
    props = params.get("properties") or {}
    required = set(params.get("required") or [])
    args = []
    for name, spec in props.items():
        typ = spec.get("type", "any") if isinstance(spec, dict) else "any"
        args.append(f"{name}{'' if name in required else '?'}: {typ}")
    desc = " ".join(str(fn.get("description", "")).split())
    if len(desc) > DESC_CHARS:
        desc = desc[:DESC_CHARS].rstrip() + "..."
    return f"{fn['name']}({', '.join(args)}) - {desc}"


def route_row(ex):
    tools = ex["tools"][:MAX_TOOLS]
    if ex["gold_tool"] and ex["gold_tool"] not in [t["name"] for t in tools]:
        tools = tools[:MAX_TOOLS - 1] + [t for t in ex["tools"] if t["name"] == ex["gold_tool"]]
    tools = tools[:]
    rng.shuffle(tools)
    lines = [f"({LETTERS[i]}) {render_tool(t)}" for i, t in enumerate(tools)]
    lines.append(f"({LETTERS[len(tools)]}) NONE - none of the tools above fits this request")
    user = ("Conversation:\n" + f"USER: {ex['query']}" + "\n\nOptions:\n" + "\n".join(lines)
            + "\n\nWhich option should happen next? Reply with the letter only.")
    names = [t["name"] for t in tools]
    answer = names.index(ex["gold_tool"]) if ex["variant"] == "tool_call" else len(names)
    return {"style": "route", "system": SYSTEM, "user": user, "n_options": len(names) + 1, "answer_index": answer}


# --- action style: Kev / When2Call's question, rendered with Kev's own renderer ---
KEV_SYSTEM = ("You are a decision model. Read the text, then answer the question by choosing exactly one of the "
              "lettered options. Reply with the option letter only.")
ACTION_CRITERIA = {"tool_call": "Call one of the available tools",
                   "request_for_info": "Ask the user for information that a tool call needs",
                   "cannot_answer": "Say it cannot help with the tools available",
                   "direct": "Answer directly without calling a tool"}


def action_row(ex):
    tools = ex["tools"][:MAX_TOOLS]
    available = [json.dumps(t) for t in tools] if tools else list("none")   # When2Call stores "none" this way
    state = {"available_tools": available, "user_message": ex["query"]}
    keys = list(ACTION_CRITERIA)
    if rng.random() < SHUFFLE_ACTION_OPTIONS:
        rng.shuffle(keys)
    lines = [f"({LETTERS[i]}) {option_text(k, ACTION_CRITERIA[k])}" for i, k in enumerate(keys)]
    user = (f"Text:\n{render(state)}\n\nQuestion: What should the assistant do next?\n\nOptions:\n" + "\n".join(lines)
            + "\n\nReply with the letter of the best option only.")
    return {"style": "action", "system": KEV_SYSTEM, "user": user, "n_options": 4,
            "answer_index": keys.index(ex["variant"])}


def general_row(r):
    # An Open-Jev decision rendered exactly like a Kev-suite question in the evaluation notebook.
    try:
        state = json.loads(r["state_json"])
    except (TypeError, json.JSONDecodeError):
        state = r["state_json"]
    target = [float(x) for x in r["target"]]
    options = list(r["options"])
    if not options or len(options) != len(target) or len(options) > 26 or max(target) < 0.6:
        return None                                   # malformed, too many options, or a genuinely soft label
    if r["kind"] == "noul":                           # stored as [no, yes]; the evaluation shows Yes first
        display = ["Yes", "No"]
        answer = 0 if target[options.index("yes")] >= target[options.index("no")] else 1
    elif r["kind"] == "choice":
        order = list(range(len(options)))
        if rng.random() < 0.5:
            rng.shuffle(order)
        display = [options[i] for i in order]
        answer = order.index(max(range(len(target)), key=target.__getitem__))
    else:                                             # score: levels lowest -> highest, order is meaning
        display = options
        answer = max(range(len(target)), key=target.__getitem__)
    lines = [f"({LETTERS[i]}) {text}" for i, text in enumerate(display)]
    if r["kind"] == "score":
        lines.append("(The options are ordered from lowest to highest.)")
    user = (f"Text:\n{render(state)}\n\nQuestion: {render(r['question']) or 'Choose the best option.'}\n\nOptions:\n"
            + "\n".join(lines) + "\n\nReply with the letter of the best option only.")
    if len(user) > 6000:
        return None
    return {"style": "action", "system": KEV_SYSTEM, "user": user, "n_options": len(display), "answer_index": answer}

## Remove anything that overlaps the evaluation sets, split and save

In [ ]:
def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()


eval_queries = set()
for cat in ("multiple", "live_multiple", "irrelevance", "live_irrelevance", "simple_python", "live_simple"):
    for line in fetch_text(BFCL_RAW + f"BFCL_v4_{cat}.json", f"bfcl_{cat}.json").splitlines():
        if line.strip():
            for m in json.loads(line)["question"][0]:
                eval_queries.add(norm(m["content"]))
for split in ("development", "test"):
    for line in fetch_text(KEV_RAW + f"evals/devtools-v1/{split}.jsonl", f"devtools_{split}.jsonl").splitlines():
        r = json.loads(line)
        if r["_meta"]["source"] == "when2call":
            eval_queries.add(norm(r["state"]["user_message"]))
for split in ("development", "test"):
    for line in fetch_text(KEV_RAW + f"evals/v4/transfer-v4/{split}.jsonl", f"transfer_{split}.jsonl").splitlines():
        eval_queries.add(norm(render(json.loads(line)["state"])))
print(len(eval_queries), "evaluation items to keep out")

rows, dropped = [], 0
for kind, exs in built.items():
    for ex in exs:
        if norm(ex["query"]) in eval_queries:
            dropped += 1
            continue
        r = route_row(ex) if rng.random() < ROUTE_SHARE[kind] else action_row(ex)
        r.update(id=f"{ex['source_id']}-{kind}", variant=kind, source_id=ex["source_id"],
                 answer_letter=LETTERS[r["answer_index"]])
        rows.append(r)
n_general = round(len(rows) * GENERAL_SHARE / (1 - GENERAL_SHARE)) if openjev_rows and GENERAL_SHARE > 0 else 0
by_source = collections.defaultdict(list)
for r in openjev_rows:
    if not re.search(EXCLUDE_GENERAL, str(r["source"]), re.I):
        by_source[r["source"]].append(r)
pools = list(by_source.values())
for pool in pools:
    rng.shuffle(pool)
added, i = 0, 0
while added < n_general and any(pools):           # round-robin over Open-Jev sources so no family dominates
    pool = pools[i % len(pools)]
    i += 1
    if not pool:
        continue
    src = pool.pop()
    try:
        state_text = norm(render(json.loads(src["state_json"])))
    except (TypeError, json.JSONDecodeError):
        state_text = norm(str(src["state_json"]))
    if state_text in eval_queries:
        dropped += 1
        continue
    g = general_row(src)
    if g is None:
        continue
    g.update(id=f"openjev-{src['id']}", variant="general", source_id=f"openjev-{src.get('group_id') or src['id']}",
             answer_letter=LETTERS[g["answer_index"]], general_source=src["source"])
    rows.append(g)
    added += 1
print(f"added {added} Open-Jev general-decision rows; dropped {dropped} rows that matched an evaluation item")

rng.shuffle(rows)
val_ids = {r["source_id"] for r in rows if rng.random() < VAL_FRACTION}
train = [r for r in rows if r["source_id"] not in val_ids]
val = [r for r in rows if r["source_id"] in val_ids]
for name, part in (("train", train), ("val", val)):
    with open(os.path.join(OUT_DIR, f"{name}.jsonl"), "w", encoding="utf-8") as f:
        for r in part:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

stats = {"train": len(train), "val": len(val), "dropped_eval_overlap": dropped,
         "by_variant": dict(collections.Counter(r["variant"] for r in train)),
         "by_style": dict(collections.Counter(r["style"] for r in train)),
         "by_style_variant": {f"{s}/{v}": n for (s, v), n in collections.Counter((r["style"], r["variant"]) for r in train).items()},
         "answer_letter": dict(sorted(collections.Counter(r["answer_letter"] for r in train).items())),
         "route_call_vs_none": dict(collections.Counter("call" if r["variant"] == "tool_call" else "NONE"
                                                        for r in train if r["style"] == "route")),
         "action_labels": dict(collections.Counter(r["variant"] for r in train
                                                   if r["style"] == "action" and r["variant"] != "general")),
         "general_sources": dict(collections.Counter(r.get("general_source") for r in train if r["variant"] == "general")),
         "approx_tokens_mean": int(sum(len(r["user"]) for r in train) / max(1, len(train)) / 3.5)}
json.dump(stats, open(os.path.join(OUT_DIR, "stats.json"), "w"), indent=1)
print(json.dumps(stats, indent=1))

In [ ]:
# Eyeball check: two examples of each kind (ask examples show what was removed)
for kind in ("request_for_info", "cannot_answer", "tool_call", "direct"):
    print("=" * 30, kind)
    for ex in built[kind][:2]:
        extra = f"   [removed: {ex['removed']!r}]" if "removed" in ex else ""
        print("-", ex["query"][:200] + extra, "| tools:", [t["name"] for t in ex["tools"]][:5])
g = next((x for x in train if x["variant"] == "general"), None)
if g:
    print("=" * 30, "general (Open-Jev) | answer", g["answer_letter"])
    print(g["user"][:800])
print("\nOne full rendered row of each style:")
for style in ("action", "route"):
    r = next(x for x in train if x["style"] == style)
    print("-" * 30, style, "| answer", r["answer_letter"], f"({r['variant']})")
    print(r["user"][:1500])